In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("parquet_to_delta")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("SparkSession démarrée.\n")

SparkSession démarrée.



26/04/03 13:21:11 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# ── Config centrale ────────────────────────────────────────────────────────────
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(_os.path.join(_os.path.dirname('__file__'), '../../..')))
from config import WAREHOUSE

PARQUET_BASE = "/opt/spark/data/parquet" if _os.path.exists("/opt/spark/data/parquet")                else _os.path.join(_os.path.dirname(WAREHOUSE), "parquet")
DELTA_BASE   = WAREHOUSE

TABLES = [
    "amazon_orders",
    "apple_app_installs",
    "apple_signin_apps",
    "google_chrome",
    "google_searches",
    "instagram_comments",
    "instagram_likes",
    "instagram_messages_meta",
    "instagram_saved",
    "netflix_views",
    "spotify_library",
    "spotify_playlists",
    "spotify_sound_capsule",
    "spotify_streams",
    "tiktok_comments",
    "tiktok_likes",
    "tiktok_messages_meta",
    "tiktok_messages_text",
    "tiktok_searches",
    "tiktok_watch",
    "twitter_likes",
    "twitter_tweets",
    "youtube_searches",
    "youtube_watch",
]

In [3]:
success = []
failed  = []

for table in TABLES:
    src  = f"{PARQUET_BASE}/{table}.parquet"
    dest = f"{DELTA_BASE}/{table}"
    try:
        df = spark.read.parquet(src)
        row_count = df.count()

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .save(dest)
        )

        print(f"[OK] {table:35s}  {row_count:>8,} lignes  →  {dest}")
        success.append(table)

    except Exception as e:
        print(f"[ERREUR] {table}: {e}")
        failed.append(table)

[OK] amazon_orders                              74 lignes  →  /opt/spark/warehouse/amazon_orders


[OK] apple_app_installs                      3,923 lignes  →  /opt/spark/warehouse/apple_app_installs
[OK] apple_signin_apps                          62 lignes  →  /opt/spark/warehouse/apple_signin_apps


[OK] google_chrome                             338 lignes  →  /opt/spark/warehouse/google_chrome


[OK] google_searches                        55,854 lignes  →  /opt/spark/warehouse/google_searches


[OK] instagram_comments                         28 lignes  →  /opt/spark/warehouse/instagram_comments


...

[OK] tiktok_watch                          234,771 lignes  →  /opt/spark/warehouse/tiktok_watch


[OK] twitter_likes                          68,534 lignes  →  /opt/spark/warehouse/twitter_likes


[OK] twitter_tweets                            319 lignes  →  /opt/spark/warehouse/twitter_tweets


[OK] youtube_searches                        4,991 lignes  →  /opt/spark/warehouse/youtube_searches


[OK] youtube_watch                          13,821 lignes  →  /opt/spark/warehouse/youtube_watch


In [4]:
print(f"\n{'='*60}")
print(f"Conversion terminée : {len(success)}/{len(TABLES)} tables OK")
if failed:
    print(f"Tables en erreur : {', '.join(failed)}")
print(f"{'='*60}")

spark.stop()


Conversion terminée : 24/24 tables OK
